# Exercises: Hypothesis Testing

**A Waiter's Tips**

The following description was retrieved from Kaggle page.

> Food servers’ tips in restaurants may be influenced by many factors, including the nature of the restaurant, size of the party, and table locations in the restaurant. Restaurant managers need to know which factors matter when they assign tables to food servers. For the sake of staff morale, they usually want to avoid either the substance or the appearance of unfair treatment of the servers, for whom tips (at least in restaurants in the United States) are a major component of pay. In one restaurant, a food server recorded the following data on all customers they served during an interval of two and a half months in early 1990. The restaurant, located in a suburban shopping mall, was part of a national chain and served a varied menu. In observance of local law, the restaurant offered to seat in a non-smoking section to patrons who requested it. Each record includes a day and time, and taken together, they show the server’s work schedule.

Acknowledgements The data was reported in a collection of case studies for business statistics. Bryant, P. G. and Smith, M (1995) Practical Data Analysis: Case Studies in Business Statistics. Homewood, IL: Richard D. Irwin Publishing

The dataset is also available through the Python package Seaborn.

In [4]:
import seaborn as sns

tips = sns.load_dataset("tips")

Here's a description of each column in the dataset:

- `total_bill`: The total bill amount, including the cost of food and drinks.
- `tip`: The tip amount given by the customer.
- `sex`: The gender of the customer (e.g., Male or Female).
- `smoker`: Whether the customer is a smoker or not (e.g., Yes or No).
- `day`: The day of the week when the transaction occurred (e.g., Sun, Sat, Thu, etc.).
- `time`: The time of day when the transaction occurred, typically categorized as Lunch or Dinner.
- `size`: The size of the party or group of customers.

**Your Task**: is to accept or reject the following hypothesis using statistical testing:

- Hypothesis $H_1$: smoking is associated with time of visit
- Hypothesis $H_2$: the bigger the group the higher the tip
- Hypothesis $H_3$: group size is different based on the time of visit
- Hypothesis $H_4$: (... come up with a hypothesis of your own ...)
- Finally, analyze if size (party size) is a **confounder**. That is, does a larger party cause a higher tip, or simply a higher bill which then leads to a higher tip?

---

In [7]:
import seaborn as sns
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

tips = sns.load_dataset("tips")
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

ct = pd.crosstab(tips["smoker"], tips["time"])
chi2, p1, dof, expected = stats.chi2_contingency(ct)

print("H1 contingency table:")
print(ct)
print(f"Chi-square p-value = {p1:.4f}")

if p1 < 0.05:
    print("Reject H0: smoking is associated with time of visit.")
else:
    print("Fail to reject H0: no significant association between smoking and time of visit.")

corr2, p2 = stats.spearmanr(tips["size"], tips["tip"])

print("\nH2:")
print(f"Spearman correlation = {corr2:.4f}")
print(f"p-value = {p2:.4e}")

if p2 < 0.05 and corr2 > 0:
    print("Reject H0: bigger groups tend to leave higher tips.")
else:
    print("Fail to reject H0: no significant positive relationship between group size and tip.")

lunch_size = tips[tips["time"] == "Lunch"]["size"]
dinner_size = tips[tips["time"] == "Dinner"]["size"]

u_stat, p3 = stats.mannwhitneyu(lunch_size, dinner_size, alternative="two-sided")

print("\nH3:")
print(f"Lunch mean size = {lunch_size.mean():.3f}")
print(f"Dinner mean size = {dinner_size.mean():.3f}")
print(f"Mann-Whitney p-value = {p3:.4f}")

if p3 < 0.05:
    print("Reject H0: group size differs by time of visit.")
else:
    print("Fail to reject H0: no significant difference in group size by time of visit.")

smoker_tip_pct = tips[tips["smoker"] == "Yes"]["tip_pct"]
nonsmoker_tip_pct = tips[tips["smoker"] == "No"]["tip_pct"]

u_stat4, p4 = stats.mannwhitneyu(smoker_tip_pct, nonsmoker_tip_pct, alternative="two-sided")

print("\nH4:")
print(f"Smoker mean tip % = {smoker_tip_pct.mean():.4f}")
print(f"Non-smoker mean tip % = {nonsmoker_tip_pct.mean():.4f}")
print(f"Mann-Whitney p-value = {p4:.4f}")

if p4 < 0.05:
    print("Reject H0: smokers and non-smokers differ in tip percentage.")
else:
    print("Fail to reject H0: no significant difference in tip percentage.")

model_size_only = smf.ols("tip ~ size", data=tips).fit()
model_with_bill = smf.ols("tip ~ size + total_bill", data=tips).fit()

print("\nConfounder analysis:")
print("\nModel 1: tip ~ size")
print(model_size_only.summary().tables[1])

print("\nModel 2: tip ~ size + total_bill")
print(model_with_bill.summary().tables[1])

H1 contingency table:
time    Lunch  Dinner
smoker               
Yes        23      70
No         45     106
Chi-square p-value = 0.4771
Fail to reject H0: no significant association between smoking and time of visit.

H2:
Spearman correlation = 0.4683
p-value = 1.0599e-14
Reject H0: bigger groups tend to leave higher tips.

H3:
Lunch mean size = 2.412
Dinner mean size = 2.631
Mann-Whitney p-value = 0.0102
Reject H0: group size differs by time of visit.

H4:
Smoker mean tip % = 0.1632
Non-smoker mean tip % = 0.1593
Mann-Whitney p-value = 0.5607
Fail to reject H0: no significant difference in tip percentage.

Confounder analysis:

Model 1: tip ~ size
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.1691      0.223      5.233      0.000       0.729       1.609
size           0.7118      0.082      8.728      0.000       0.551       0.872

Model 2: tip ~ size + to